### VGAE w/ Hierarchical representations

- Dim-reduction on features (`z`: representation, `u`: interpretability)<br> $z \in \mathbb{R}^{N' \times 10}$, $u \in \mathbb{R}^{N' \times 1}$
- Reconstruction w/ $\sigma(z \cdot z^T)$ for graph reconstruction, regularize $u$ w/ CyCIF graph prior
- TODO: Benchmark against K-means clustering & PCA

In [ ]:
import os
import gc
import sys
import pickle

import numpy as np
import cupy as cp
import pandas as pd
import scanpy as sc

import torch
import tifffile
import torch.nn as nn

import networkx as nx
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy import sparse
from torch_geometric import utils as pyg_utils

torch.manual_seed(42)

In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, plot, gen_graph, configs, dataset, zonation
import vgae, vgae_sbm, model_train, model_eval

from torch_geometric.loader import DataLoader

### Running VGAE

Load DESI normalized iamge & CyCIF prior computation

In [ ]:
# Load DESI normed image
data_path = '../data/integrated/'
desi_imgs = [tifffile.imread(os.path.join(data_path, 'desi', f))
             for f in sorted(os.listdir(os.path.join(data_path, 'desi')))
             if f[-3:] == 'tif']

# Load full-size u_prior from CyCIF image (graph-based heat diffusion)
u_priors = [tifffile.imread(os.path.join(data_path, 'zonation_prior', f))
            for f in sorted(os.listdir(os.path.join(data_path, 'zonation_prior')))
            if f[-3:] == 'tif']

ndimy, ndimx = desi_imgs[0].shape[-2:]
yy, xx = np.meshgrid(np.arange(desi_imgs[0].shape[-2]), np.arange(desi_imgs[0].shape[-1]))
desi_coords = np.array([yy.flatten(), xx.flatten()]).T

gc.collect()

# Testing robustness w/ different graph connections
# G_desi = gen_graph.construct_graph(desi_coords, k=8, r=10)  
# plot.disp_network(desi_img[7], G_desi, figsize=(12, 12), node_size=0.8, edge_width=0.5)

In [ ]:
# histogram of DESI expressions
# Compute feature matrix from desi_image [C, X, Y] --> [X*Y, C]
idx = 10
nchans = desi_imgs[0].shape[0]
desi_feature_mat = desi_imgs[idx].transpose(2,1,0).reshape(-1, nchans) 
desi_feature_mat = utils.norm_features(desi_feature_mat)

# idx = np.random.choice(np.arange(ndimy*ndimx), 1000, replace=False)
# plt.figure(figsize=(6, 2))
# plt.hist(desi_feature_mat[idx, :].flatten(), bins=30, edgecolor='white')
# plt.title('Histogram of DESI expressions')
# plt.show()

Create sub-graphs with mini-batches

In [ ]:
# DEBUG model
from importlib import reload
reload(vgae)
reload(vgae_sbm)
reload(utils)
reload(plot)
reload(model_train)
reload(model_eval)
reload(configs)
reload(dataset)

In [ ]:
# Debug for correctness: train on a sample individual image
ims_loader = dataset.DESIGraphDataset(data_path = '../data/integrated/desi_debug/',
                                      prior_path = '../data/integrated/zonation_prior_debug/',
                                      k=8, r=10, n_subgraphs=8)
ims_data = ims_loader.load_graphs()
ims_train_dl = DataLoader(ims_data, shuffle=True)

# ims_loader = dataset.DESIGraphDataset(data_path='../data/integrated/desi/',
#                                       prior_path='../data/integrated/zonation_prior/',
#                                       k=8, r=10,
#                                       n_subgraphs=8)
# ims_data = ims_loader.load_graphs()

# # Saving subgraphs
# # subgraphs = []
# # for x in ims_data:
# #     subgraph = pyg_utils.to_networkx(x, node_attrs=['pos'], to_undirected=True)
# #     subgraphs.append(subgraph)
# # del x, subgraph

# ims_train, ims_test = torch.utils.data.random_split(ims_data, [0.5, 0.5],
#                                                     generator=torch.Generator().manual_seed(42))
# # Split training & validation set
# ims_train_dl = DataLoader(ims_train)
# ims_test_dl = DataLoader(ims_test)

Model training:

In [ ]:
del model, optimizer

In [ ]:
# del model, optimizer
lr = 0.01
n_epochs = 100
device = torch.device('cuda')

train_configs = configs.set_train_configs(data_path=None, 
                                          n_epochs=n_epochs, 
                                          gamma=0.98,
                                          lr=lr)

model_configs = configs.set_model_configs(device=device,
                                          beta=0.5,
                                          c_in=nchans,
                                          c_hidden=8,
                                          c_latent=1,
                                          alpha=5,
                                          pu_scale=1.)

model = vgae_sbm.SparseVGAE(model_configs)
optimizer = torch.optim.Adam(model.parameters(), lr=train_configs.lr)

In [ ]:
losses, nlls, sparse_losses, kls, signs = model_train.train(model, ims_train_dl, train_configs)

In [ ]:
fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10, 18))

ax1.plot(np.arange(n_epochs), losses)
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Total Loss')

ax2.plot(np.arange(n_epochs), nlls)
ax2.set_xlabel('Epochs')
ax2.set_ylabel('NLLs')

ax3.plot(np.arange(n_epochs), sparse_losses)
ax3.set_xlabel('Epochs')
ax3.set_ylabel('Orthogonal regularization')

ax4.plot(np.arange(n_epochs), kls)
ax4.set_xlabel('Epochs')
ax4.set_ylabel('KLs')

ax5.plot(np.arange(n_epochs), signs)
ax5.set_xlabel('Epochs')
ax5.set_ylabel('Sign')

plt.tight_layout()
plt.show()

In [ ]:
torch.save(model, '../data/desi/res/model_int.pt')

In [ ]:
# Graph reconstruction checks on full graph

idx = 10
desi_feature_mat = desi_imgs[idx].transpose(2, 1, 0).reshape(-1, nchans)
desi_feature_mat = utils.norm_features(desi_feature_mat)

G_desi = gen_graph.construct_graph(desi_coords, k=8, r=10)  
A = nx.adjacency_matrix(G_desi).toarray()

model.eval()
with torch.no_grad():
    latent = model_eval.eval(model, G_desi, desi_feature_mat)
    recon = model.decoder(latent)

z = latent.qz.detach().cpu().numpy()
u_posterior = latent.qu.detach().cpu().numpy()
A_hat = recon.A_hat.detach().cpu().numpy()
X_hat = recon.px_loc.detach().cpu().numpy()

Predictive checks:

In [ ]:
# plot distributions of u, z & A_hat
plt.figure(figsize=(4, 2))
plt.hist(u_posterior.flatten().squeeze(), edgecolor='white', bins=10)
plt.show()

In [ ]:
plt.figure()
sample_idx = np.random.choice(np.arange(desi_feature_mat.shape[0]), 2000, replace=False)

fig, ax = plt.subplots()
ax.scatter(desi_feature_mat[sample_idx, :].flatten(), X_hat[sample_idx, :].flatten(), s=0.1)
ax.spines[['right', 'top']].set_visible(False)
ax.get_xaxis().tick_bottom()
ax.get_yaxis().tick_left()
ax.set_title('Feature matrix reconstruction')
plt.show()

In [ ]:
plt.imshow(A[:256, :256], cmap='magma')
plt.show()
plt.imshow(A_hat[:256, :256], cmap='magma')
plt.show()

In [ ]:
# Soft membership assignment (latent.qb):
g = sns.clustermap(latent.qb.detach().cpu().numpy(), col_cluster=False)
g.ax_heatmap.axes.set_title('Node cluster assignment', fontsize=20)

In [ ]:
z_norm = z / np.sum(z, axis=1, keepdims=True)
g = sns.clustermap(z, col_cluster=False)
g.ax_heatmap.axes.set_title('Node cluster strength', fontsize=20)
# sns.clustermap(z)

In [ ]:
# Visuzlize spatial distribution of Node (n) ->module (z) assignments

def disp_module_spatial(z, height, ncol=4, 
                        panel_size=3, cmap='turbo'):
    assert z.shape[0] % height == 0,\
         "2D spatial plots should have int height & width"
    n_factors = z.shape[-1]  # dim: [N x factor]
    nrow = n_factors // ncol if n_factors % ncol == 0 else n_factors // ncol + 1
    width = z.shape[0] // height
    
    idx = 0
    fig, axes = plt.subplots(nrow, ncol, figsize=((panel_size+0.2)*ncol, panel_size*nrow), dpi=100)
    for r in range(nrow):
        for c in range(ncol):
            axes[r, c].axis('off')
            if idx < n_factors:
                axes[r, c].imshow(z[:, idx].reshape(height, width).T, cmap=cmap)
                axes[r, c].set_title('Factor {}'.format(idx))
            idx += 1
    plt.tight_layout()
    plt.show()

In [ ]:
disp_module_spatial(z, height=desi_imgs[idx].shape[-1],
                    panel_size=4, ncol=3)

In [ ]:
# Correlations btw Zs
fig, ax = plt.subplots(figsize=(6, 6), dpi=200)
sns.heatmap(np.corrcoef(z_norm.T), cmap='coolwarm', ax=ax, square=True)
fig.suptitle('Correlations btw z-factors')
plt.show()


In [ ]:
# Interprete metabolites (x) -> module (z) assignments
W_xz = model.decoder.z_to_xloc[0].weight.detach().cpu().numpy()

g = sns.clustermap(W_xz, vmin=0, cmap='magma', col_cluster=False)
g.ax_heatmap.axes.set_title('Metabolite-to-factor assignments', y=1.3, fontsize=20)
plt.show()

In [ ]:
# Visualize UMAPs of z & x
adata_z = sc.AnnData(z)
for i in range(z.shape[1]):
    label = 'z'+str(i)
    adata_z.obs[label] = z_norm[:, i]

sc.pp.neighbors(adata_z)
sc.tl.umap(adata_z)

In [ ]:
sc.pl.umap(adata_z, color=adata_z.obs.columns)

In [ ]:
adata_x = sc.AnnData(desi_feature_mat)

# Label DESI features w/ latents?
for i in range(z.shape[1]):
    label = 'z'+str(i)
    adata_x.obs[label] = z_norm[:, i]

sc.pp.pca(adata_x)
sc.pp.neighbors(adata_x)
sc.tl.umap(adata_x)

In [ ]:
# sc.pl.pca_variance_ratio(adata_x)
sc.pl.umap(adata_x, color=adata_x.obs.columns)
# del adata_z, adata_x

In [ ]:
# Visualize prior & posterior `u`:

# Note: coords saved in [x,y]-index in graph, but in [i,j]-index in image!!:
# transpose 2D image before flatten
plot.disp_network_embedding(desi_imgs[idx].mean(0), G_desi, embedding=u_priors[idx].T.flatten(), alpha=0.5, 
                       node_size=10, edge_width=0.5, fontsize=64,
                       figsize=(15, 15), title='U prior (CyCIF)')

plot.disp_network_embedding(desi_imgs[idx].mean(0), G_desi, embedding=u_posterior.flatten(), alpha=0.5, 
                       node_size=10, edge_width=0.5, fontsize=64,
                       figsize=(15, 15), title='U posterior')

In [ ]:
# Create discretized zonation maps from posterior `u`
u_posterior_map = u_posterior.reshape(-1, desi_imgs[idx].shape[-1]).T
u_posterior_discretized_map = utils.infer_zones(u_posterior_map, nbins=10)

plt.figure()
plt.imshow(u_posterior_discretized_map, cmap='turbo')
plt.colorbar()
plt.title('Discretised trajectory \n Hierarchical VGAE', fontsize=15)
plt.axis('off')
plt.show()

In [ ]:
# Save U posterior
np.save('../data/desi/res/u_posterior.npy', u_posterior)

### Result Analysis

#### 2D 

Visualize DESI gradients sorted on a sample posterior `U`:

In [ ]:
idx = 10

# Load DESI m/z labels:
ifile = open('../data/integrated/desi/DESI_slice_48.ome.tif', 'rb')
ion_labels = IO.load_ome_labels(ifile, '../data/integrated/desi/DESI_slice_48.ome.tif')
ifile.close()

In [ ]:
# Visualize DESI features sorted by prior temperature
desi_feature_mat_sorted = desi_feature_mat.reshape(-1, desi_feature_mat.shape[-1])  # Coord x Feature
desi_feature_mat_sorted = desi_feature_mat_sorted[np.argsort(u_priors[idx].T.flatten())]

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Prior Temperature', nbins=10)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Prior Temperature', nbins=20)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                        ion_labels, 
                        title='Prior Temperature', nbins=100)


In [ ]:
# Visualize DESI features sorted by POSTERIOR temperature
desi_feature_mat_sorted = desi_feature_mat.reshape(-1, desi_feature_mat.shape[-1])  # Coord x Feature
desi_feature_mat_sorted = desi_feature_mat_sorted[np.argsort(u_posterior.squeeze())] # sort by posterior temp

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels,
                         title='Posterior Temperature', nbins=10)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Posterior Temperature', nbins=20)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Posterior Temperature', nbins=100)

In [ ]:
nbins = 100
# Feature x nbins
desi_binned_grads, desi_binned_stds = utils.get_binned_features(desi_feature_mat_sorted, nbins=nbins) 
desi_binned_grads, desi_binned_stds = desi_binned_grads.T, desi_binned_stds.T

# Sort metabolites based on argmax values along the PV->CV binned trajectory
argmax_grad = desi_binned_grads.argmax(1)

sorted_ion_labels = np.array(ion_labels)[np.argsort(argmax_grad)]
sorted_desi_img = desi_imgs[idx][np.argsort(argmax_grad), ...]
desi_binned_grads = desi_binned_grads[np.argsort(argmax_grad), :]
desi_binned_stds = desi_binned_stds[np.argsort(argmax_grad), :]

desi_binned_grads_df = pd.DataFrame(desi_binned_grads,  
                                    index=sorted_ion_labels,
                                    columns=['bin' + str(i) for i in range(nbins)])

display(desi_binned_grads_df.head())
desi_binned_grads_df.to_csv('../data/desi/res/metabolite_binned_gradients.csv', index=True)
del nbins

In [ ]:
plot.disp_desi_gradients(desi_binned_grads_df.values.T, 
                         sorted_ion_labels, cluster_ions=False,
                         title='Posterior Temperature', nbins=100)

In [ ]:
# Smoothed gradient plots of indiv. features:
from scipy.stats import linregress

nbins = 100
desi_gradient_ts = []  # Fitted slope of the features along PV -> CV trajectory

for feature_mean, feature_std, label in zip(desi_binned_grads, desi_binned_stds, sorted_ion_labels):
    # plot.disp_desi_gradient(feature_mean, feature_std, title=label)
    ts = linregress(np.linspace(-1, 1, nbins), feature_mean)
    desi_gradient_ts.append(ts)

del label, feature_mean, feature_std

In [ ]:
# Summarize expression gradienst along PV->CV trajectory
desi_gradient_slopes = [ts.slope for ts in desi_gradient_ts]
desi_gradient_pvals = [ts.pvalue for ts in desi_gradient_ts]
desi_gradient_categories = []
for slope, pval in zip(desi_gradient_slopes, desi_gradient_pvals):
    if pval >= 0.05:
        desi_gradient_categories.append('Uncertain')
    elif slope > 0:
        desi_gradient_categories.append('+')
    else:
        desi_gradient_categories.append('-')


In [ ]:
valid_slopes = [slope 
                for (slope, p) in zip(desi_gradient_slopes, desi_gradient_pvals)
                if p < 0.05]

plt.figure(figsize=(5, 2))
plt.hist(valid_slopes, edgecolor='white', bins=10)
plt.xlabel(r'Fitted slope $\beta$')
plt.suptitle('Metabolite gradients along PV->CV trajectory')
plt.show()

del valid_slopes

In [ ]:
# summarize metabolite gradient as slopes
gradient_df = pd.DataFrame({
    'Label': sorted_ion_labels,
    'Slope': desi_gradient_slopes,
    'Category': desi_gradient_categories,
    'p-val': desi_gradient_pvals
})

gradient_df.set_index('Label', inplace=True)
gradient_df.to_csv('../data/desi/res/metabolite_gradients.csv', index=True)
gradient_df.head()

Check annotated metabolites:

SEAM:
- FA: `m/z 255.22, 279.22, 281.23`
- Kupffer cell metabolites: `m/z 134.04, 180.89` (Adanine: `m/z 134.04`)
- Glutamine: `m/z 145.04`

In [ ]:
annot_labels = [
    '865.49527 m/z ± 50 mDa',
    '267.08191 m/z ± 50 mDa',
    '295.22855 m/z ± 50 mDa',
    '261.14629 m/z ± 50 mDa', 
    '253.21007 m/z ± 25.16 mDa', 
    '301.22061 m/z ± 50 mDa',
    '303.24218 m/z ± 50 mDa',
    '346.06614 m/z ± 50 mDa',
    '609.52634 m/z ± 50 mDa',
    '699.51653 m/z ± 50 mDa'
]


In [ ]:
annot_indices = []
for i, (feature_mean, feature_std, label) in enumerate(zip(desi_binned_grads, desi_binned_stds, sorted_ion_labels)):
    if label in annot_labels:
        annot_indices.append(i)

        # Trajectory plot
        plot.disp_desi_gradient(feature_mean, feature_std, title=label)

        # DESI feature image
        plt.figure(figsize=(6, 6), dpi=100)
        plt.imshow(sorted_desi_img[i], cmap='magma')
        plt.title(label)
        plt.show()

        
del label, feature_mean, feature_std

In [ ]:
# Learnt metabolites->module assignments
sorted_W_xz = W_xz[np.argsort(argmax_grad)]
g = sns.clustermap(sorted_W_xz[annot_indices], 
                   yticklabels=sorted_ion_labels[annot_indices],
                   xticklabels=['Module_' + str(i) for i in range(z.shape[1])],
                   cmap='coolwarm',
                   col_cluster=False,
                   figsize=(6, 6))

plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45)
g.ax_heatmap.axes.set_title('Annotated metabolites & Module weights', y=1.3, fontsize=15)
plt.show()

Example associatations btw soft-assignments & Module weights:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(z_norm[:, 2].reshape(-1, desi_imgs[idx].shape[-1]).T, cmap='turbo')
ax1.set_title('Module 1')
ax1.axis('off')

ax2.imshow(sorted_desi_img[annot_indices[1]], cmap='magma')  # Metabolite 267
ax2.set_title('Representative PV metabolites')
ax2.axis('off')
plt.show()

In [ ]:
# Learnt metabolites->module assignments
sorted_W_xz = W_xz[np.argsort(argmax_grad)]

g = sns.clustermap(np.corrcoef(sorted_W_xz[annot_indices]),
                   xticklabels=[],
                   yticklabels=sorted_ion_labels[annot_indices],
                   figsize=(6, 4),
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of module assignments\n btw annotated DESI features', y=1.3, fontsize=20)
plt.show()

In [ ]:
# Binned-expression correlations
g = sns.clustermap(desi_binned_grads_df.loc[annot_labels].T.corr(),
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n btw annotated DESI features', y=1.3, fontsize=20)
plt.show()

In [ ]:
# Raw-expresssion correlations 
g = sns.clustermap(np.corrcoef(desi_feature_mat_sorted[:, annot_indices].T),
                    xticklabels=sorted_ion_labels[annot_indices],
                    yticklabels=sorted_ion_labels[annot_indices],
                    cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of raw-expressions\n btw annotated DESI features', y=1.3, fontsize=20)
plt.show()

**TODO: spatial correlation metrics (e.g. Moran's I)**

#### 3D integration

- Cluster metabolites with similar trajectory
- Check 3D patterning for annotated metabolites (whether necessary to perform any smoothing to account for noises)

Infer latents across slices:

In [ ]:
# Load data:
# ...

In [ ]:
# Load model:
# ...

In [ ]:
# # Common graph structure for each L-slice
G_desi = gen_graph.construct_graph(desi_coords, k=8, r=10)  
A = nx.adjacency_matrix(G_desi).toarray()

b_list = []  # Binary spatial cluster assignment
z_list = []  # Spatial node cluster strength assignment
u_post_list = []  # Trajectory assignment 
desi_features_3d = []
module_weights = model.decoder.z_to_xloc[0].weight.detach().cpu().numpy()

with torch.no_grad():
    for i, desi_img in enumerate(desi_imgs):
        desi_features = desi_img.transpose(2, 1, 0).reshape(-1, nchans)
        desi_features = utils.norm_features(desi_features)
        desi_features_3d.append(desi_features)
        
        latent = model_eval.eval(model, G_desi, desi_features)
        b = latent.qb.detach().cpu().numpy()
        z = latent.qz.detach().cpu().numpy()
        u_post = latent.qu.detach().cpu().numpy()
        w = model.decoder.z_to_xloc[0].weight.detach().cpu().numpy()

        b_list.append(b)
        z_list.append(z)
        u_post_list.append(u_post)
        module_weights_list.append(w)

del desi_img, desi_features, latent, b, z, u_post, w

In [ ]:
u_post_list_2d = [u_post.reshape(ndimy, ndimx).T for u_post in u_post_list]
plot.disp_chans(u_post_list_2d, cmap='turbo', title=r'Zonation posterior ($u$)')
#del u_post_list_2d

In [ ]:
u_post_discrete_2d = [utils.infer_zones(u_post, nbins=10) for u_post in u_post_list_2d]
plot.disp_chans(u_post_discrete_2d, cmap='coolwarm', title=r'Discretized zonation posterior ($u$)')

In [ ]:
tifffile.imwrite('../data/desi/res/u_posteriors.tif', np.array(u_post_list_2d))
tifffile.imwrite('../data/desi/res/u_posteriors_discretized.tif', np.array(u_post_discretized_2d))


Check 3D trajectory of representative metabolites:

In [ ]:
desi_features_3d = np.array(desi_features_3d)  # [L, Y*X, C]
desi_features_unsorted = desi_features_3d.transpose(2, 0, 1).reshape(nchans, -1)  # [C, L*Y*X]
desi_features_full = np.zeros_like(desi_features_unsorted)  # [C, L*Y*X]

indices = np.argsort(np.array(u_post_list).squeeze().flatten())
for i, chan in enumerate(desi_features_unsorted):
    desi_features_full[i] = chan[indices]

del desi_features_unsorted, indices, chan

In [ ]:
# Visualize binned trajectory:
plot.disp_desi_gradients(desi_features_full.T, 
                         ion_labels, 
                         title='Posterior Temperature (3D)', nbins=100)

In [ ]:
# Compute binned trajectory
nbins = 100
desi_grads, desi_stds = utils.get_binned_features(desi_features_full.T , nbins=nbins)  
desi_grads, desi_stds = desi_grads.T, desi_stds.T  # [C, #bins]

# Sort metabolites based on argmax values along the PV->CV binned trajectory
argmax_grad = desi_grads.argmax(1)

sorted_ion_labels = np.array(ion_labels)[np.argsort(argmax_grad)]
desi_grads = desi_grads[np.argsort(argmax_grad), :]
desi_stds = desi_stds[np.argsort(argmax_grad), :]

desi_grads_df = pd.DataFrame(desi_grads,  
                             index=sorted_ion_labels,
                             columns=['bin' + str(i) for i in range(nbins)])

display(desi_grads_df.head())
# desi_grads_df.to_csv('../data/desi/res/metabolite_binned_gradients_3d.csv', index=True)
del nbins

In [ ]:
annot_indices = []
for i, (feature_mean, feature_std, label) in enumerate(zip(desi_grads, desi_stds, sorted_ion_labels)):
    if label in annot_labels:
        annot_indices.append(i)

        # Trajectory plot
        plot.disp_desi_gradient(feature_mean, feature_std, title=label + ' (3D)')
        
del label, feature_mean, feature_std

SEAM m/z annotation:
- FA: `m/z 227.18, 251.20, 253.20, 255.22, 279.22, 281.23, 283.35`
- Kupffer cell metabolites: `m/z 134.04, 180.89` (Adanine: `m/z 134.04`)
- Glutamine: `m/z 145.04`
- Guanine?: `m/z 150.99`
- Phosphocholine-CH3?: `m/z 168.05`

In [ ]:
# Annotation of known m/z reported from other papers?? 
for i, label in enumerate(sorted_ion_labels):
    if '283' in label:
        print(i, label)

In [ ]:
ion_idx = 179
plot.disp_desi_gradient(desi_grads[ion_idx], desi_stds[ion_idx], title=sorted_ion_labels[ion_idx] + ' (3D)')
del ion_idx

In [ ]:
# Binned-expression correlations
g = sns.clustermap(desi_binned_grads_df.loc[annot_labels].T.corr(),
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n btw annotated DESI features', y=1.3, fontsize=20)
plt.show()

In [ ]:
# Raw-expresssion correlations 
g = sns.clustermap(np.corrcoef(desi_features_full[annot_indices, :]),
                    xticklabels=sorted_ion_labels[annot_indices],
                    yticklabels=sorted_ion_labels[annot_indices],
                    cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of raw-expressions\n btw annotated DESI features', y=1.3, fontsize=20)
plt.show()

Group modules by (1). Module weight assignment; (2).binned trajectories via hierarchical clustering:

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

In [ ]:
# Cluster by Module weight assignment `W_xz`
linked_W_xz = linkage(module_weights, method='ward')

fig, ax = plt.subplots(figsize=(8, 20), dpi=500)
dendrogram(linked_W_xz, labels=ion_labels, color_threshold=3, 
           orientation='left', distance_sort='descending',
           show_leaf_counts=True, ax=ax)
plt.show()
fig.savefig('../data/desi/res/metabolite_module_W_xz.pdf', bbox_inches='tight', dpi=300)

In [ ]:
# Louvain cluster by trajectory `u`

# adata_mz_grads = sc.AnnData(desi_grads_df)  # [C, #bins]
# sc.pp.neighbors(adata_mz_grads, n_neighbors=10)
# sc.tl.louvain(adata_mz_grads)
# sc.tl.umap(adata_mz_grads)
# sc.pl.umap(adata_mz_grads, color='louvain', s=30)

In [ ]:
sns.clustermap(desi_grads_df, col_cluster=False, method='ward', cmap='coolwarm')

In [ ]:
linked_grads = linkage(desi_grads_df, method='ward')

fig, ax = plt.subplots(figsize=(6, 18), dpi=300)
dendrogram(linked_grads, labels=desi_grads_df.index, color_threshold=3, 
           orientation='left', distance_sort='descending',
           show_leaf_counts=True, ax=ax)
plt.show()
# fig.savefig('../data/desi/res/metabolite_module_gradients.pdf', bbox_inches='tight', dpi=300)

In [ ]:
# Tree-cut for clusters
max_d = 3
clusters = fcluster(linked_grads, max_d, criterion='distance')
desi_grads_df.insert(0, 'Cluster', clusters)
display(desi_grads_df.head())
desi_grads_df.to_csv('../data/desi/res/mz_modules_3d.csv', index=True)

### Benchmarks
Baseline methods on full 3D data

#### GASTON

In [ ]:
import gaston
from gaston import process_NN_output, dp_related, cluster_plotting

In [ ]:
gaston_model, A, S = process_NN_output.process_files('../data/desi/res/GASTON_output/')
# S = np.flip(S, axis=1)

In [ ]:
n_layers = 9
u_gaston, gaston_labels = dp_related.get_isodepth_labels(gaston_model, A, S, n_layers)
u_gaston = u_gaston.reshape(-1, desi_img.shape[-1])  # Convert to dim=[i, j]
u_gaston = -u_gaston  # tmp: Convert to relatively correct PV->CV trajectory

In [ ]:
plt.figure()
plt.imshow(u_gaston, cmap='turbo')
plt.colorbar()
plt.axis('off')
plt.title('GASTON isodepth', fontsize=15)
plt.show()

In [ ]:
# Visualize DESI features sorted by GASTON trajectory
desi_feature_mat_gaston = desi_feature_mat.reshape(-1, desi_feature_mat.shape[-1])  # Coord x Feature
desi_feature_mat_gaston = desi_feature_mat_gaston[np.argsort(u_gaston.T.flatten())]

plot.disp_desi_gradients(desi_feature_mat_gaston, 
                         ion_labels, 
                         title='GASTON isodepth', nbins=10)

plot.disp_desi_gradients(desi_feature_mat_gaston, 
                         ion_labels, 
                         title='GASTON isodepth', nbins=20)

plot.disp_desi_gradients(desi_feature_mat_gaston, 
                         ion_labels, 
                         title='GASTON isodepth', nbins=100)


In [ ]:
# Plot example metabolites along GASTON isodepth

nbins = 100
# Feature x nbins
desi_gaston_grads, desi_gaston_stds = utils.get_binned_features(desi_feature_mat_gaston, nbins=nbins) 
desi_gaston_grads, desi_gaston_stds = desi_gaston_grads.T, desi_gaston_stds.T

# Sort metabolites based on argmax values along the PV->CV binned trajectory
argmax_grad = desi_gaston_grads.argmax(1)

sorted_gaston_ion_labels = np.array(ion_labels)[np.argsort(argmax_grad)]
sorted_gaston_desi_img = desi_img[np.argsort(argmax_grad), ...]
desi_gaston_grads = desi_gaston_grads[np.argsort(argmax_grad), :]
desi_gaston_stds = desi_gaston_stds[np.argsort(argmax_grad), :]

desi_gaston_grads_df = pd.DataFrame(desi_gaston_grads,  
                                    index=sorted_gaston_ion_labels,
                                    columns=['bin' + str(i) for i in range(nbins)])

display(desi_binned_grads_df.head())
del nbins

In [ ]:
for i, (feature_mean, feature_std, label) in enumerate(zip(desi_gaston_grads, desi_gaston_stds, sorted_gaston_ion_labels)):
    if label in annot_labels:
        # Trajectory plot
        plot.disp_desi_gradient(feature_mean, feature_std, title=label)

        # DESI feature image
        plt.figure(figsize=(6, 6), dpi=100)
        plt.imshow(sorted_gaston_desi_img[i], cmap='magma')
        plt.title(label)
        plt.show()

        
del label, feature_mean, feature_std
del desi_gaston_grads, desi_gaston_stds, sorted_gaston_ion_labels

#### PCA

In [ ]:
# Unmute if X run in the previous section
adata_list = []
u_pcs = []
for desi_img in desi_imgs:
    desi_features = desi_img.transpose(2, 1, 0).reshape(-1, nchans)
    adata_x = sc.AnnData(desi_features)
    sc.pp.pca(adata_x, n_comps=3)

    u_pca = adata_x.obsm['X_pca']
    u_pc1 = u_pca[:, 0]

    adata_list.append(adata_x)
    u_pcs.append(u_pc1)

u_pcs_discrete = [utils.infer_zones(u_pc.reshape(ndimy, ndimx).T, nbins=10) for u_pc in u_pcs]
del u_pca, u_pc1

In [ ]:
# for i in range(8):
#     plt.figure()
#     plt.imshow(u_pca[:, i].reshape(-1, desi_img.shape[-1]).T, cmap='turbo')
#     plt.title('PC{}'.format(i+1), fontsize=15)
#     plt.axis('off')
#     plt.show()

In [ ]:
plot.disp_chans([u_pc.reshape(ndimy, ndimx).T for u_pc in u_pcs], cmap='turbo', title='PC1')
plot.disp_chans(u_pcs_discrete, cmap='turbo', title='PC1 (discrete)')

In [ ]:
desi_features_3d = np.array(desi_features_3d)
desi_features_unsorted = desi_features_3d.transpose(2, 0, 1).reshape(nchans, -1)  # Feature x Coord
desi_features_full = np.zeros_like(desi_features_unsorted)

indices = np.argsort(np.array(u_pcs).squeeze().flatten())
for i, chan in enumerate(desi_features_unsorted):
    desi_features_full[i] = chan[indices]

del desi_features_unsorted, indices, chan

In [ ]:
# Visualize DESI trajectory represented as PC0
plot.disp_desi_gradients(desi_features_full.T, 
                         ion_labels, 
                         title='PC1 isodepth', nbins=100)


In [ ]:
# Compute binned trajectory
nbins = 100
desi_pc_grads, desi_pc_stds = utils.get_binned_features(desi_features_full.T , nbins=nbins)  
desi_pc_grads, desi_pc_stds = desi_pc_grads.T, desi_pc_stds.T  # [C, #bins]

# Sort metabolites based on argmax values along the PV->CV binned trajectory
argmax_grad = desi_pc_grads.argmax(1)

sorted_ion_labels = np.array(ion_labels)[np.argsort(argmax_grad)]
sorted_desi_img = desi_imgs[idx][np.argsort(argmax_grad), ...]
desi_pc_grads = desi_pc_grads[np.argsort(argmax_grad), :]
desi_pc_stds = desi_pc_stds[np.argsort(argmax_grad), :]

desi_pc_grads_df = pd.DataFrame(desi_pc_grads,  
                                index=sorted_ion_labels,
                                columns=['bin' + str(i) for i in range(nbins)])

del nbins

In [ ]:
for i, (feature_mean, feature_std, label) in enumerate(zip(desi_pc_grads, desi_pc_stds, sorted_ion_labels)):
    if label in annot_labels:
        # Trajectory plot
        plot.disp_desi_gradient(feature_mean, feature_std, title=label + ' (PC1 3D)')
        
del label, feature_mean, feature_std
del desi_pc_grads, desi_pc_stds

In [ ]:
# Binned-expression correlations
g = sns.clustermap(desi_pc_grads_df.loc[annot_labels].T.corr(),
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n btw annotated DESI features', y=1.3, fontsize=20)
plt.show()

#### Clustering
e.g. K-means / Louvain / Leiden

In [ ]:
idx = 10
desi_features = desi_imgs[idx].transpose(2, 1, 0).reshape(-1, nchans)
adata_x = sc.AnnData(desi_features)

sc.pp.pca(adata_x)
sc.pp.neighbors(adata_x)
sc.tl.louvain(adata_x)
adata_x.obs.louvain = adata_x.obs.louvain.astype(np.uint8)

In [ ]:
# Use cluster assignment to represent spatial node modules
u_louvain = adata_x.obs.louvain.to_numpy().reshape(ndimy, ndimx).T
u_louvain_one_hot = [(u_louvain == i).astype(np.uint8)
                     for i in np.sort(adata_x.obs.louvain.unique())]  

plot.disp_chans(u_louvain_one_hot, cmap='magma')

#### LDA

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation as LDA

In [ ]:
idx = 10
desi_features = desi_imgs[idx].transpose(2, 1, 0).reshape(-1, nchans)
lda = LDA(n_components=5, random_state=42)
lda.fit(desi_features)

In [ ]:
theta = lda.transform(desi_features)
beta = lda.components_.T

In [ ]:
sns.clustermap(theta)

In [ ]:
sns.clustermap(beta)

In [ ]:
for i in range(lda.n_components):
    plt.figure()
    plt.imshow(theta[:, i].reshape(ndimy, ndimx).T, cmap='magma')
    plt.show()

In [ ]:
sns.clustermap(beta.T)

---